# Train a pile detector for Brick Sorter

This teaches a detector to pull a **pile** of parts and minifigure bits apart, so each item can be identified. Your brick detector learned from bricks spread out on a table; this one learns from piles made out of **your own parts**.

How it works:
1. You photograph your parts **spread out** (the scanner already handles that).
2. Every item is cut out of those photos with its exact outline.
3. The cut-outs are dropped on top of each other, thousands of times, to make piles. The script places every item, so it knows where each one is, and every pile comes with its boxes already drawn.
4. A detector trains on those piles on your graphics card.

### What you need

Put these next to this notebook (they're in the brick-sorter folder):
- `make_piles.py`
- `brick_sorter.py`

Then fill these folders inside `brick-detector` (step 1 creates them):

| Folder | What to put in it | How many |
|---|---|---|
| `piles/spread` | Photos of parts **spread out** with a gap between each, on a plain background, shot from above, about as far away as you'll shoot your piles. Mix of heads, torsos, legs, hair, hats, accessories and bricks. Flip everything over and shoot again to get both sides. | 20–40 photos of 20–30 items each, so **300 or more items** |
| `piles/empty` | *(Optional)* Photos of empty surfaces you might use: the grey cloth, a table, white paper, in different light. | 3–10 |
| `piles/test` | Real pile photos to check the result. **Not** used for training. | 3–5 |
| `brick-detector` itself | *(Optional)* The training zip from the web app (Settings → Training photos → Download). Its photos, with every item you kept boxed, are added to the training data. | Any |

Phone photos: send them to the PC as JPG (AirDrop, or set Camera → Formats → Most Compatible). HEIC files are skipped.

Run the cells in order. Making the piles takes a few minutes; training about 1–2 hours on your 3080.

## 1. Set up

In [ ]:
import os, sys, shutil, importlib
from pathlib import Path
import torch

HERE = Path.cwd()
missing = [f for f in ("make_piles.py", "brick_sorter.py") if not (HERE / f).exists()]
if missing:
    raise SystemExit(f"Put {' and '.join(missing)} in {HERE} (next to this notebook), then run this cell again.")
sys.path.insert(0, str(HERE))
import make_piles
importlib.reload(make_piles)

WORK = HERE / "brick-detector"
PILES = WORK / "piles"
for d in ("spread", "empty", "test"):
    (PILES / d).mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: training will be very slow")
for d in ("spread", "empty", "test"):
    print(f"{PILES / d}: {len(make_piles.list_images(PILES / d))} photos")

## 2. Settings

`MODEL_SIZE`: `s` makes a detector small enough to run on your phone in the web app (about 37 MB). `m` is a bit more accurate but about 80 MB, better on the PC only.

In [ ]:
MODEL_SIZE = "s"      # n, s (good for the phone), m (more accurate, PC)
IMGSZ = 960           # same as your brick detector
COUNT = 4000          # piles to make; 3000-6000 is plenty
EPOCHS = 60           # stops early if it stops improving
BATCH = -1            # -1 picks the largest batch that fits the graphics card
RUN_NAME = f"piles-{MODEL_SIZE}-{IMGSZ}"

## 2b. Add the photos you saved in the web app (optional)

Put the `brick-sorter-training-….zip` files you downloaded from the app (Settings → Training photos) in `brick-detector`. Every scan photo in them comes with a box around each item you kept, so they go straight into training as real examples. Leave `USE_AS_SPREAD` off unless those photos were all taken on a plain background: turned on, items are also cut out of them to build more piles.

In [ ]:
import zipfile
USE_AS_SPREAD = False
APP_DETECTION = []
zips = sorted(set(WORK.glob("brick-sorter-training*.zip")) | set(HERE.glob("brick-sorter-training*.zip")))
for z in zips:
    dest = WORK / "app-training" / z.stem
    if not dest.exists():
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
    det = dest / "detection"
    n = len(make_piles.list_images(det / "images")) if det.exists() else 0
    if n:
        APP_DETECTION.append(det)
        if USE_AS_SPREAD:
            for img in make_piles.list_images(det / "images"):
                shutil.copy(img, PILES / "spread" / f"app_{z.stem[-10:]}_{img.name}")
    parts = len(list((dest / "parts").rglob("*.jpg"))) if (dest / "parts").exists() else 0
    print(f"{z.name}: {n} full photos with boxes (used), {parts} single-part photos (kept for later)")
if not zips:
    print(f"No brick-sorter-training*.zip in {WORK}. That's fine: this step is optional.")

## 3. Cut the items out

Then **look at the preview**. Each picture should be one item with the background removed. Delete any file in `brick-detector/piles/cutouts` that is two items stuck together, a shadow, or a speck (the file names are under each picture), then carry on. If lots of items are missing or cut badly, retake those photos with more space between items, or change `sensitivity` (higher finds faint parts, lower ignores shadows).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

summary = make_piles.extract(PILES / "spread", PILES, PILES / "empty", sensitivity=1.0)
plt.figure(figsize=(16, 12)); plt.imshow(mpimg.imread(PILES / "cutouts_preview.jpg")); plt.axis("off"); plt.show()

## 4. Make the piles

Uses several processor cores. Check the preview: every visible item should have a box. Items mostly hidden under others are left unboxed on purpose, so the detector doesn't learn to guess at them.

In [ ]:
data_yaml = make_piles.generate(PILES, COUNT, extra_real=APP_DETECTION)
plt.figure(figsize=(16, 9)); plt.imshow(mpimg.imread(PILES / "piles_preview.jpg")); plt.axis("off"); plt.show()

## 5. Train

Starts from a model pre-trained on everyday photos. Checkpoints are saved to `brick-detector/runs` after every epoch. Training stops by itself if it stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO(f"yolo11{MODEL_SIZE}.pt")
results = model.train(
    data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    project=str(WORK / "runs"), name=RUN_NAME, exist_ok=True,
    patience=15, cos_lr=True, close_mosaic=10,
    degrees=0, flipud=0.5, fliplr=0.5, scale=0.3, mosaic=0.5,
    workers=2, plots=True,
)

**If training stopped part-way** (PC restarted, cell interrupted): run step 1 and step 2, then this cell. It carries on from the last saved epoch. The piles from step 4 are still on disk, so you don't need to make them again.

In [ ]:
from ultralytics import YOLO
last = WORK / "runs" / RUN_NAME / "weights" / "last.pt"
if not last.exists():
    print("No checkpoint yet at", last)
else:
    try:
        model = YOLO(str(last))
        results = model.train(resume=True)
    except AssertionError:
        print("Training already finished, nothing to resume. Carry on to step 6.")

## 6. Try it on your real piles

This is the real test: your photos in `piles/test`, which it has never seen. `CONF` is how sure it must be to box something: lower it if items are missed, raise it if it boxes things that aren't items. Your old brick detector is shown alongside for comparison, if `lego-detector.pt` or `lego-detector.onnx` is in `brick-detector`.

In [ ]:
from ultralytics import YOLO
CONF = 0.3

best = WORK / "runs" / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best))
m = model.val(data=str(PILES / "dataset" / "data.yaml"), imgsz=IMGSZ, plots=False, verbose=False)
print(f"On made-up piles it hasn't trained on: mAP50 {m.box.map50:.3f}, recall {m.box.mr:.3f} (above 0.9 is very good)")

old_path = next((p for p in (WORK / "lego-detector.pt", WORK / "lego-detector.onnx", HERE / "lego-detector.onnx") if p.exists()), None)
old = YOLO(str(old_path), task="detect") if old_path else None
tests = make_piles.list_images(PILES / "test")
if not tests:
    print(f"Put a few photos of real piles in {PILES / 'test'} and run this cell again.")
for p in tests:
    r = model.predict(str(p), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    panels = [(r, "Pile detector")]
    if old is not None:
        panels.append((old.predict(str(p), imgsz=IMGSZ, conf=CONF, verbose=False)[0], "Old brick detector"))
    fig, axes = plt.subplots(1, len(panels), figsize=(9 * len(panels), 8))
    for ax, (res, title) in zip(([axes] if len(panels) == 1 else axes), panels):
        ax.imshow(res.plot(labels=False)[:, :, ::-1]); ax.set_title(f"{title}: {len(res.boxes)} items"); ax.axis("off")
    plt.suptitle(p.name); plt.tight_layout(); plt.show()

## 7. Save it

Saves `pile-detector.pt` (for the script) and `pile-detector.onnx` (for the web app) in `brick-detector`.

- **Script:** `python brick_sorter.py scan pile.jpg --mode mixed --model pile-detector.pt --debug`
- **Web app:** under **Part finder** choose **Brick detector**, then **Load detector (.onnx)** and pick `pile-detector.onnx`. It replaces the old one in that browser; **Remove detector** and load the old file to switch back.
- **On your phone:** send me the `.onnx` file (or add it to your GitHub repository) and I can make the website load it by itself.

In [ ]:
out_pt = WORK / "pile-detector.pt"
shutil.copy(best, out_pt)
print("Saved:", out_pt)
try:
    onnx_path = Path(YOLO(str(best)).export(format="onnx", imgsz=IMGSZ, simplify=True))
    shutil.copy(onnx_path, WORK / "pile-detector.onnx")
    print("Saved:", WORK / "pile-detector.onnx", f"({(WORK / 'pile-detector.onnx').stat().st_size / 1e6:.0f} MB)")
except Exception as e:
    print("ONNX export skipped:", e, "(pip install onnx onnxslim, then run this cell again)")